In [17]:
from read_model_runs import read_model_runs

# Ler dados completos
question_long_df = \
    read_model_runs('../../data/processed/model-runs')



In [18]:
question_long_df.head(2)

,model_name,firac,language,is_correct,pdf_filename,question_id,materia,oab_test_id,oab_question_id,chosen_option,...,Facts,Issue,Rule,Application,Conclusion,rule_count,fact_count,response_time_seconds,datetime,tema
0,gemini-2.0-flash-lite,FILA_,portuguese,True,oab-153.pdf,oab-153.pdf-007,DIREITO CIVIL,II,7,C,...,['A existência de um regime de solidariedade p...,Qual a correta aplicação das regras do regime ...,"Art. 276 do Código Civil, Art. 279 do Código C...",Ao analisar a responsabilidade em caso de fale...,A alternativa C está correta. Em caso de perda...,4,5,4.855925,NaN,DIREITO DAS OBRIGAÇÕES
1,gemini-2.0-flash-lite,FILA_,portuguese,True,oab-153.pdf,oab-153.pdf-014,DIREITO CIVIL,II,14,D,...,['João prometeu transferir a propriedade de um...,Qual o regime jurídico aplicável à obrigação d...,Art. 235 do Código Civil,"Considerando que uma coisa certa, objeto de pr...","De acordo com o artigo 235 do Código Civil, no...",1,3,2.210483,NaN,DIREITO DAS OBRIGAÇÕES


In [19]:
print('shape:', question_long_df.shape)
print('# unique questions:', question_long_df['question_id'].nunique())
print("models:", question_long_df['model_name'].unique())
print("firacs:", question_long_df['firac'].unique())



shape: (90606, 32)
# unique questions: 3230
models: ['gemini-2.0-flash-lite' 'gemini-2.5-flash-lite' 'gemma-3-12b-it'
 'gemma-3-27b-it' 'gemma-3-4b-it' 'gemma-3n-e2b-it' 'gemma-3n-e4b-it']
firacs: ['FILA_' 'FIL__' 'FI___' '_____' 'FIR__' 'F____' 'unstructured' 'FIRAC']


In [20]:
import pandas as pd

# Total de question_id únicos
total_questions = question_long_df["question_id"].nunique()

# Contagem de question_id únicos por FIRAC
firac_counts = (
    question_long_df
    .groupby("firac")["question_id"]
    .nunique()
    .reset_index(name="n_question_id")
)

# Percentual de question_id por FIRAC
firac_counts["pct_question_id"] = (
    firac_counts["n_question_id"] / total_questions * 100
)

# Ordenar por percentual (do menor para o maior)
firac_counts = firac_counts.sort_values(
    by="pct_question_id",
    ascending=True
).reset_index(drop=True)

firac_counts


,firac,n_question_id,pct_question_id
0,F____,1493,46.222910
1,FIRAC,2940,91.021672
2,FI___,3142,97.275542
3,_____,3142,97.275542
4,FIL__,3192,98.823529
5,FIR__,3213,99.473684
6,FILA_,3221,99.721362
7,unstructured,3229,99.969040


In [21]:
import pandas as pd

# Total de question_id únicos no dataset inteiro
total_questions = question_long_df["question_id"].nunique()

# Contagem de question_id únicos por FIRAC
firac_counts = (
    question_long_df
    .groupby("firac")["question_id"]
    .nunique()
    .reset_index(name="n_question_id")
)

# Percentual de question_id por FIRAC
firac_counts["pct_question_id"] = (
    firac_counts["n_question_id"] / total_questions * 100
)

# -------------------------------------------------------
# NOVO: cobertura por combinação FIRAC × model_name
# -------------------------------------------------------
firac_model_counts = (
    question_long_df
    .groupby(["firac", "model_name"])["question_id"]
    .nunique()
    .reset_index(name="n_question_id_model")
)

# Percentual de cobertura por FIRAC × modelo
firac_model_counts["pct_question_id_model"] = (
    firac_model_counts["n_question_id_model"] / total_questions * 100
)

result_df = (
    firac_model_counts
    .merge(
        firac_counts[["firac"]],
        on="firac",
        how="left"
    )
    .sort_values(
        by="pct_question_id_model",
        ascending=False
    )
    .reset_index(drop=True)
)

result_df


,firac,model_name,n_question_id_model,pct_question_id_model
0,unstructured,gemma-3-12b-it,3229,99.969040
1,unstructured,gemma-3-27b-it,3229,99.969040
2,unstructured,gemma-3-4b-it,3229,99.969040
3,unstructured,gemma-3n-e4b-it,3227,99.907121
4,FILA_,gemma-3-27b-it,3218,99.628483
5,FILA_,gemma-3-12b-it,3217,99.597523
6,FIR__,gemma-3-27b-it,3213,99.473684
7,FIL__,gemma-3-27b-it,3190,98.761610
8,FIL__,gemma-3-12b-it,3189,98.730650
9,FIL__,gemma-3-4b-it,3171,98.173375


In [22]:
from read_model_runs import filter_complete_questions

models = ['gemma-3-27b-it', 'gemma-3-12b-it', 'gemma-3-4b-it']
firacs = ['FILA_', 'FIR__', 'FI___', 'FIL__', '_____', 'unstructured']
question_long_df, question_wide_df, model_order, firac_order = filter_complete_questions(question_long_df, models, firacs)

print('shape:', question_long_df.shape)
print('# unique questions:', question_long_df['question_id'].nunique())
print("model order:", model_order)
print("firac order:", firac_order)


shape: (43596, 32)
# unique questions: 2422
model order: ['gemma-3-4b-it', 'gemma-3-12b-it', 'gemma-3-27b-it']
firac order: ['_____', 'unstructured', 'FIL__', 'FI___', 'FIR__', 'FILA_']


In [23]:
question_wide_df

,question_id,model_name,_____,unstructured,FIL__,FI___,FIR__,FILA_
0,oab-1.pdf-002,gemma-3-12b-it,False,True,True,True,True,True
1,oab-1.pdf-002,gemma-3-27b-it,False,True,True,True,True,True
2,oab-1.pdf-002,gemma-3-4b-it,True,True,True,True,True,True
3,oab-1.pdf-003,gemma-3-12b-it,False,False,False,False,True,True
4,oab-1.pdf-003,gemma-3-27b-it,False,False,True,True,True,True
...,...,...,...,...,...,...,...,...
7261,oab-99.pdf-010,gemma-3-27b-it,True,True,True,True,True,True
7262,oab-99.pdf-010,gemma-3-4b-it,True,True,True,True,True,True
7263,oab-99.pdf-011,gemma-3-12b-it,True,True,True,True,True,True
7264,oab-99.pdf-011,gemma-3-27b-it,True,True,True,True,True,True


In [24]:
# TODO: adicionar gemma-3n-4
# TODO: rodar FIRAC
# TODO: consistencia: de acerto e de opção